# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata information
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
if hasattr(dataset.metadata, 'keywords'):
    print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Review available Record Sets and their fields (`@id` provided for reference).

Let's list all record sets defined in the Croissant metadata and examine their structure.

In [ ]:
# Get all record sets by their @id
record_sets = dataset.record_sets

print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"@id: {rs.id}")
    print(f"Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - Field: {getattr(field, 'name', None)} / @id: {field.id}, DataType: {getattr(field, 'data_type', None)}")
    print('\n')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the Record Set and Field `@id`s from the overview.

In [ ]:
# Collect all record set @id's to extract
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Optional: limit to first 1000 for demonstration
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set {rs_id} with columns: {df.columns.tolist()}")

# For illustration, pick the first record set for EDA below
selected_record_set_id = record_set_ids[0] if record_set_ids else None

# Show head of this DataFrame
if selected_record_set_id:
    print(f"\nFirst 5 records from: {selected_record_set_id}")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we select a numeric field (by `@id`), filter records, normalize, and group by a categorical field.

In [ ]:
# Select example numeric and group fields (adjust these to match actual field @id's in your dataset)
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    fields = dataset.record_set(selected_record_set_id).fields
    numeric_fields = [f for f in fields if getattr(f, 'data_type', None) in ('Float', 'Integer', 'Number')]
    group_fields = [f for f in fields if getattr(f, 'data_type', None) in ('Text',)]

    numeric_field_id = numeric_fields[0].id if numeric_fields else df.columns[0]
    group_field_id = group_fields[0].id if group_fields else df.columns[0]

    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    # Ensure field exists and is numeric
    if numeric_field_id in df.columns:
        # Remove non-numeric coercion issues
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() == df[numeric_field_id].mean() else 0  # Fallback to 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())
        # Group
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
    else:
        print(f"No suitable numeric field found in DataFrame columns: {df.columns.tolist()}")

## 5. Visualization
Visualize the distribution of a selected numeric field and, if possible, its grouping by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    if numeric_field_id in df.columns:
        # Histogram of the numeric field
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()
        # Boxplot grouping if possible
        if group_field_id in df.columns:
            plt.figure(figsize=(8, 4))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.xticks(rotation=45)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a Croissant-annotated FAIR^2 dataset using the `mlcroissant` library. We examined its record sets, filtered and transformed numeric fields using their `@id`, and visualized key features. This workflow can be adapted for any Croissant dataset to quickly understand its structure and perform custom analysis.

For further analysis, refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) or iterate through other available record sets and fields.